# Explore Readings

`main.py` loads `data/raw/readings.csv` into the `Readings` repository
(backed by `app.db`). This notebook picks up from there: it reads `app.db`
directly and writes a derived summary to `data/processed/`, standalone,
outside a running `Project`.

## Using a Repository standalone

`@Repository` returns the decorated class unchanged — no base class, no
injected methods, no hidden state. It only registers the class and its
`Entity` relationship so the `Context` *can* wire it up inside a running
`Project`. Nothing stops you from instantiating it directly and passing
its dependencies by hand, exactly like any other Python object — which
is what a notebook kernel needs, since it runs outside the resolved
object graph.

Here `Readings` (`app/repositories.py`) normally receives its `SQLite`
dependency via DI. Standalone, we build that same dependency chain
ourselves: a `Config` with just the bit `SQLite` needs, then
`SQLite(config)`, then `Readings(sqlite)`.

In [ ]:
import sys

sys.path.insert(0, "..")

from app.repositories import Readings
from app.storages import SQLite
from opendataframework import Config

config = Config({"sqlite": {"path": "../app.db"}})
sqlite = SQLite(config)
readings = Readings(sqlite)

readings.all()

## Deriving a summary into `data/processed/`

Average `celsius` per `sensor`, written to `data/processed/summary.csv` —
the raw CSV in `data/raw/` never gets touched.

In [ ]:
import csv
from collections import defaultdict

by_sensor = defaultdict(list)
for reading in readings.all():
    by_sensor[reading.sensor].append(reading.celsius)

averages = {sensor: sum(values) / len(values) for sensor, values in by_sensor.items()}

with open("../data/processed/summary.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["sensor", "average_celsius"])
    for sensor, average in averages.items():
        writer.writerow([sensor, round(average, 2)])

averages